# Region similarity between mouse (MOSTA) and human (HESTA)

How similar is gene expression in the same annotated region (heart, brain, liver, ...) of a mouse and a human embryo? The unit of comparison is the **embryo**: all slices of an embryo found in `get_data_dir()` are pooled (e.g. `CS17_E1` = `CS17_E1S8` + `CS17_E1S9`).

**Profiles.** For every embryo and region, the raw counts of all bins with that label are summed over the embryo's slices (pseudobulk), scaled to counts per 10k (CP10k) and log1p transformed. Raw counts are used because `.X` differs between the atlases (log-normalized in HESTA, not in MOSTA). `Profile` then chooses what is compared:
- `relative` (default): each gene's value minus its mean over all regions of the same embryo. This is a region-specificity profile: positive means the gene is higher in this region than in the rest of the embryo. Two regions are similar when the *same genes are up and down* in both.
- `absolute`: the log-CP10k values themselves. Housekeeping genes dominate, so every pair of regions scores high (ρ ≈ 0.8-0.9) and differences are small.

**Genes.** `Gene set` decides which genes enter and how mouse genes are matched to human genes:
- `1:1 orthologs`: one-to-one human-mouse orthologs (`mgw/orthologs.py`, Ensembl). Renamed genes are matched correctly.
- `shared symbols`: genes whose symbol exists in both species, ignoring case (`Sox2` = `SOX2`). Close to the orthologs, but homonyms are not always orthologs.
- `all genes`: every gene of either species, matched by symbol ignoring case; a gene missing in one species counts as 0 there.

Two filters follow. `Min CP10k`: a gene is kept if it reaches this level in at least one region of the embryos being compared (genes at 0 everywhere would only add ties). `Top variable`: keeps the genes with the highest variance across regions, chosen per species and joined (0 keeps all).

**Similarity.** Spearman's ρ between two profiles (distance = 1 − ρ). It compares ranks, so it does not depend on sequencing depth or on capture differences between the two atlases. Pearson and cosine similarity are offered for comparison; on `relative` profiles cosine is nearly the same as Pearson.

**Baseline.** A cross-species ρ only means something next to the ρ between two embryos of the *same* species and stage. Those pairs are computed with the same settings and shown in grey.

**Labels.** HESTA (`celltype`) and MOSTA (`annotation`) use different vocabularies. `harmonized` maps both to organ classes (`mgw/labels.py`, the map of `demos/human_mouse_align_merged.ipynb`) and leaves out blood, cavity and labels without a counterpart. `original` keeps the labels and matches them by name, ignoring case; only cavity and extraembryonic tissue are left out. Regions with fewer than `MIN_BINS` bins in an embryo are left out.

| Tab | Shows |
|---|---|
| Matched regions | ρ of every embryo pair per region: cross-species pairs (stage matched or not) next to the within-species baseline. The main answer. |
| Region × region | every mouse region against every human region for one embryo pair. A strong diagonal means the regions correspond. |
| Embryo × embryo | one region in every embryo, clustered. Do embryos group by species or by stage? |
| Gene scatter | one region in one embryo pair, one point per gene, with the genes that agree and disagree most. With *All regions*: each gene's correlation across the shared regions between mouse and human, i.e. which genes follow the same (positive) or the opposite (negative) regional pattern. With *all embryos* as Mouse and Human, every stage-matched pair of stages contributes, so time enters too; *Component* separates regional from temporal agreement. |
| Pair summary | per embryo pair: matched vs mismatched ρ, and how often a region's best match is the region of the same name. |
| Cross-species genes | every gene in mouse against every gene in human over paired regions: gene pairs and clusters that behave alike in both species. |

**Cross-species gene correlation.** The last tab compares genes instead of regions. A correlation between a mouse gene and a human gene needs observations that exist in both species; bins and embryos do not, annotated regions do. With `Pairing` = `stage matched`, each observation is one harmonized region at one stage pair of `STAGE_QUALITY` (e.g. heart at E10.5/CS14-15): the relative profiles of that stage's embryos are averaged per region in each species, and the regions present in both are kept. `all stages` averages over all embryos instead (one observation per region; coarser, but uses every embryo). Over these paired observations, `X[i, j]` is the similarity (`Similarity` measure) of gene *i* in mouse and gene *j* in human:
- the **diagonal** (*i* = *j*) asks whether a gene behaves the same in both species;
- **off the diagonal**, mouse gene *i* behaves like human gene *j*;
- **blocks** after clustering are groups of genes that rise and fall together across the two species.

Genes must pass `Min CP10k` in both species and are ranked by the smaller of their two variances across regions; `Cross genes` sets how many. This tab always uses harmonized labels and relative profiles (`Labels`, `Profile` and `Top variable` do not apply). With only 15-40 paired observations, sizeable correlations arise by chance, so every result is compared with **shuffled pairings**: the human regions are permuted within each stage pair 50 times.

The pseudobulk of each slice is cached in `<data dir>/pseudobulk/`, so slices are read only once.

In [ ]:
import re
import sys
import time
from itertools import combinations

sys.path.append("..")

import h5py
import ipywidgets as w
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import scipy.sparse as sp
from anndata.io import read_elem
from IPython.display import HTML, clear_output, display
from plotly.subplots import make_subplots
from scipy.cluster.hierarchy import fcluster, leaves_list, linkage
from scipy.spatial.distance import squareform
from scipy.stats import rankdata

from mgw.config import get_data_dir
from mgw.labels import NOT_SCORED, STAGE_QUALITY, harmonize
from mgw.orthologs import load_one2one, restrict_to_orthologs

# ----------------------------
# Parameters
# ----------------------------
DATA_DIR = get_data_dir()  # asks on first run, then remembered per machine
CACHE_DIR = DATA_DIR / "pseudobulk"

# per dataset: species, sub-folder, file pattern, name suffix, raw-count layer, region-label column in obs
DATASETS = {
    "MOSTA": dict(species="mouse", subdir="mosta", pattern="*.MOSTA.h5ad", suffix=".MOSTA",
                  counts_layer="count", label_col="annotation"),
    "HESTA": dict(species="human", subdir="hesta", pattern="*_HESTA.h5ad", suffix="_HESTA",
                  counts_layer="counts", label_col="celltype"),
}
LABEL_FALLBACKS = ("annotation", "celltype", "primaryanno", "simpleanno")
SLICE_RE = re.compile(r"^(?P<stage>.+)_(?P<embryo>E\d+)S\d+$")  # <stage>_E<k>S<n>; embryo = <stage>_E<k>

DROP_LABELS = ["Cavity", "Extraembryonic Tissue"]  # never a region (original labels; harmonized drops NOT_SCORED)
MIN_BINS = 30            # a region needs this many bins in an embryo to get a profile
TARGET_SUM = 1e4         # CP10k
BLOCK_ROWS = 20_000      # bins read at a time when summing a slice
CACHE_VERSION = 1

# viewer defaults
DEFAULT_GENE_SET = "orthologs"
DEFAULT_TOP_VARIABLE = 2000
DEFAULT_MIN_CP10K = 0.1
N_TABLE = 15             # genes per table in the gene scatter
N_LABELLED = 5           # genes per group labelled in the gene scatter
ALL_REGIONS = "All regions"  # gene scatter: correlate each gene across all shared regions
N_REGION_SHUFFLES = 200  # shuffled region orders for the null of the all-regions gene correlation
ALL_EMBRYOS = "all embryos (by stage)"  # gene scatter, all regions: every stage-matched pair of stages
COMPONENTS = {"all": "regions + time", "regions": "regions", "time": "time"}
CROSS_GENES = [100, 200, 300, 500]  # genes on each axis of the cross-species matrix (n x n)
DEFAULT_CROSS_GENES = 300
DEFAULT_MODULES = 8
N_PERMUTATIONS = 50      # shuffled pairings for the null
N_PAIRS = 25             # rows in the top cross pairs table

# colours (categorical slots in fixed order; diverging blue <-> red with a grey midpoint)
C_MATCHED, C_OTHER, C_BASELINE = "#2a78d6", "#eb6834", "#898781"
C_HUMAN_UP, C_GENE = "#1baf7a", "#c3c2b7"
INK, MUTED = "#0b0b0b", "#898781"
DIVERGING = [[0.0, "#104281"], [0.25, "#5598e7"], [0.5, "#f0efec"], [0.75, "#ec8a86"], [1.0, "#a32424"]]
SEQUENTIAL = [[0.0, "#cde2fb"], [0.5, "#3987e5"], [1.0, "#0d366b"]]

## Embryos

Every `.h5ad` file named `<stage>_E<k>S<n>` is one slice of embryo `<stage>_E<k>`. Other files (e.g. the HESTA 3D reconstructions) are skipped.

In [ ]:
def _natural(s):
    return [int(t) if t.isdigit() else t for t in re.split(r"(\d+)", s)]


def discover_slices():
    rows = []
    for ds, cfg in DATASETS.items():
        files = [p for p in (DATA_DIR / cfg["subdir"]).glob(cfg["pattern"]) if not p.name.startswith(".")]
        for p in sorted(files, key=lambda p: _natural(p.name)):
            name = p.name.removesuffix(".h5ad").removesuffix(cfg["suffix"])
            m = SLICE_RE.match(name)
            if m is None:
                print(f"skipped {p.name}: not <stage>_E<k>S<n>")
                continue
            rows.append(dict(dataset=ds, species=cfg["species"], stage=m["stage"],
                             embryo=f"{m['stage']}_{m['embryo']}", section=name, path=p))
    return pd.DataFrame(rows)


SLICES = discover_slices()
if SLICES.empty or SLICES.species.nunique() < 2:
    raise FileNotFoundError(f"Need MOSTA and HESTA .h5ad files under {DATA_DIR}")
print(f"{len(SLICES)} slices of {SLICES.embryo.nunique()} embryos under {DATA_DIR}:")
for (sp_, emb), g in SLICES.groupby(["species", "embryo"], sort=False):
    print(f"  {sp_:<6} {emb:<12} {', '.join(g.section)}")

## Pseudobulk per region

The raw counts of a slice are summed per region label, reading `BLOCK_ROWS` bins at a time so that large slices fit in memory. The sums of all slices of an embryo are then added up; a gene missing from a slice counts as 0 there.

In [ ]:
def read_labels(h, col):
    # region labels as a Categorical; MOSTA files use the legacy encoding (int codes + obs/__categories/<col>)
    obs = h["obs"]
    if "__categories" in obs and col in obs["__categories"]:
        cats = [c.decode() if isinstance(c, bytes) else str(c) for c in obs["__categories"][col][:]]
        return pd.Categorical.from_codes(obs[col][:], cats)
    return pd.Categorical(read_elem(obs[col]))


def _gene_names(h):
    var = h["var"]
    return np.asarray(read_elem(var[var.attrs["_index"]])).astype(str)


def _onehot(codes, n_regions):
    keep = codes >= 0  # code -1: no label
    return sp.csr_matrix((np.ones(keep.sum()), (codes[keep], np.flatnonzero(keep))), shape=(n_regions, codes.size))


def _region_sums(group, codes, n_regions):
    # sum the rows of the stored count matrix per region code (regions x genes)
    if group.attrs.get("encoding-type") != "csr_matrix":
        return np.asarray((_onehot(codes, n_regions) @ sp.csr_matrix(read_elem(group))).todense())
    n_obs, n_var = (int(s) for s in group.attrs["shape"])
    indptr = group["indptr"][:]
    out = np.zeros((n_regions, n_var))
    for start in range(0, n_obs, BLOCK_ROWS):
        stop = min(start + BLOCK_ROWS, n_obs)
        lo, hi = indptr[start], indptr[stop]
        X = sp.csr_matrix((group["data"][lo:hi], group["indices"][lo:hi], indptr[start:stop + 1] - lo),
                          shape=(stop - start, n_var))
        out += (_onehot(codes[start:stop], n_regions) @ X).toarray()
    return out


def slice_pseudobulk(row):
    # summed raw counts per region of one slice (regions x genes) and the bins per region; cached on disk
    cfg = DATASETS[row.dataset]
    cache = CACHE_DIR / f"{row.dataset}_{row.section}.pkl"
    mtime = row.path.stat().st_mtime
    if cache.exists():
        saved = pd.read_pickle(cache)
        if saved["version"] == CACHE_VERSION and saved["mtime"] == mtime:
            return saved["sums"], saved["n_bins"]
    with h5py.File(row.path, "r") as h:
        genes = _gene_names(h)
        label_col = next(c for c in (cfg["label_col"], *LABEL_FALLBACKS) if c in h["obs"])
        labels = read_labels(h, label_col)
        codes = np.asarray(labels.codes)
        sums = _region_sums(h["layers"][cfg["counts_layer"]], codes, len(labels.categories))
    regions = [str(c).strip() for c in labels.categories]
    n_bins = pd.Series(np.bincount(codes[codes >= 0], minlength=len(regions)), index=regions)
    sums = pd.DataFrame(sums, index=regions, columns=genes)
    if sums.columns.has_duplicates:
        sums = sums.T.groupby(level=0).sum().T
    used = (n_bins > 0).to_numpy()
    sums, n_bins = sums[used], n_bins[used]
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    pd.to_pickle(dict(version=CACHE_VERSION, mtime=mtime, sums=sums, n_bins=n_bins), cache)
    return sums, n_bins


EMBRYOS = {}  # embryo -> species, stage, sections, sums (regions x genes, raw counts), n_bins (per region)
for emb, g in SLICES.groupby("embryo", sort=False):
    t0 = time.time()
    parts = [slice_pseudobulk(r) for r in g.itertuples()]
    sums = pd.concat([s for s, _ in parts]).fillna(0).groupby(level=0).sum()
    n_bins = pd.concat([n for _, n in parts]).groupby(level=0).sum()
    EMBRYOS[emb] = dict(species=g.species.iloc[0], stage=g.stage.iloc[0], sections=list(g.section),
                        sums=sums, n_bins=n_bins.loc[sums.index])
    print(f"{emb:<12} {len(g)} slice(s), {int(n_bins.sum()):>7,} bins, {sums.shape[0]:>2} regions, "
          f"{sums.shape[1]:>6,} genes ({time.time() - t0:.1f}s)")

MOUSE = [e for e, d in EMBRYOS.items() if d["species"] == "mouse"]
HUMAN = [e for e, d in EMBRYOS.items() if d["species"] == "human"]

## Gene sets, profiles and similarity

Mouse genes are mapped to a common key: the upper-cased symbol (`all`, `shared`) or the human ortholog (`orthologs`). The counts of each embryo are summed per key, scaled by the region's total count over *all* of its genes (so the gene set does not change the scale) and log1p transformed. All embryos share one gene axis per gene set.

In [ ]:
ORTHO = load_one2one()
GENE_SETS = {"orthologs": "1:1 orthologs", "shared": "shared symbols", "all": "all genes"}


def _universe(species):
    genes = set()
    for e, d in EMBRYOS.items():
        if d["species"] == species:
            genes.update(d["sums"].columns)
    return genes


def _key_maps():
    # gene set -> species -> {symbol: common key}
    mouse_genes, human_genes = _universe("mouse"), _universe("human")
    upper = {"mouse": {g: g.upper() for g in mouse_genes}, "human": {g: g.upper() for g in human_genes}}
    both = set(upper["mouse"].values()) & set(upper["human"].values())
    h, m = restrict_to_orthologs(human_genes, mouse_genes, ORTHO)
    return {
        "all": upper,
        "shared": {s: {g: k for g, k in upper[s].items() if k in both} for s in upper},
        "orthologs": {"human": dict(zip(h, h)), "mouse": dict(zip(m, h))},
    }


KEY_MAPS = _key_maps()
GENE_AXIS = {gs: pd.Index(sorted(set(km["mouse"].values()) | set(km["human"].values()))) for gs, km in KEY_MAPS.items()}
MOUSE_SYMBOL = {gs: {k: g for g, k in km["mouse"].items()} for gs, km in KEY_MAPS.items()}  # key -> mouse symbol
for gs, axis in GENE_AXIS.items():
    print(f"{GENE_SETS[gs]:<15} {len(axis):>6,} genes")

# label shown for a lower-cased original label (first spelling seen, mouse first)
DISPLAY = {}
for e in MOUSE + HUMAN:
    for r in EMBRYOS[e]["sums"].index:
        DISPLAY.setdefault(r.lower(), r)
DROP_LOWER = {l.lower() for l in DROP_LABELS}


def region_table(emb, labels):
    # raw counts per region of one embryo under `labels` ("original" or "harmonized") and the bins per region
    d = EMBRYOS[emb]
    sums, n_bins = d["sums"], d["n_bins"]
    if labels == "harmonized":
        key = pd.Index(harmonize(sums.index)[0])
        keep = ~key.isin(NOT_SCORED)
    else:
        low = sums.index.str.lower()
        key = pd.Index([DISPLAY[l] for l in low])
        keep = ~low.isin(DROP_LOWER)
    sums = sums[keep].groupby(key[keep]).sum()
    n_bins = n_bins[keep].groupby(key[keep]).sum()
    big = n_bins >= MIN_BINS
    return sums[big.to_numpy()], n_bins[big]


_PROFILES = {}


def log_profiles(labels, gene_set):
    # {embryo: regions x genes log1p(CP10k) on GENE_AXIS[gene_set]}, and {embryo: bins per region}
    key = (labels, gene_set)
    if key not in _PROFILES:
        axis, out, bins = GENE_AXIS[gene_set], {}, {}
        for e, d in EMBRYOS.items():
            sums, n_bins = region_table(e, labels)
            lib = sums.sum(axis=1).to_numpy()
            keys = sums.columns.map(KEY_MAPS[gene_set][d["species"]])  # NaN: not in this gene set
            mapped = sums.loc[:, keys.notna()].T.groupby(keys[keys.notna()]).sum().T
            mapped = mapped.reindex(columns=axis, fill_value=0.0)
            out[e] = pd.DataFrame(np.log1p(mapped.to_numpy() / lib[:, None] * TARGET_SUM).astype(np.float32),
                                  index=mapped.index, columns=axis)
            bins[e] = n_bins
        _PROFILES[key] = out, bins
    return _PROFILES[key]


_TOP_VARIABLE = {}


def top_variable(labels, gene_set, n):
    # boolean mask on the gene axis: the n genes with the highest variance across regions, per species, joined
    key = (labels, gene_set, n)
    if key not in _TOP_VARIABLE:
        P, _ = log_profiles(labels, gene_set)
        mask = np.zeros(len(GENE_AXIS[gene_set]), bool)
        for embs in (MOUSE, HUMAN):
            var = np.mean([P[e].to_numpy().var(axis=0) for e in embs], axis=0)
            mask[np.argsort(var)[::-1][:n]] = True
        _TOP_VARIABLE[key] = mask
    return _TOP_VARIABLE[key]


def select_genes(embryos, s):
    # mask on the gene axis: reaches min CP10k in a region of one of `embryos`, and among the top variable genes
    P, _ = log_profiles(s["labels"], s["gene_set"])
    thr = np.log1p(s["min_cp10k"])
    keep = np.zeros(len(GENE_AXIS[s["gene_set"]]), bool)
    for e in embryos:
        X = P[e].to_numpy()
        keep |= ((X >= thr) if thr > 0 else (X > 0)).any(axis=0)
    if s["top_variable"]:
        keep &= top_variable(s["labels"], s["gene_set"], s["top_variable"])
    if keep.sum() < 10:
        raise ValueError(f"only {keep.sum()} genes pass the filters; lower 'Min CP10k' or raise 'Top variable'")
    return keep


def profiles(emb, keep, s):
    # regions x selected genes for one embryo: log1p(CP10k), centred per gene over the embryo's regions if relative
    P, _ = log_profiles(s["labels"], s["gene_set"])
    X = P[emb].to_numpy()[:, keep].astype(np.float64)
    if s["profile"] == "relative":
        X = X - X.mean(axis=0)
    return pd.DataFrame(X, index=P[emb].index, columns=GENE_AXIS[s["gene_set"]][keep])


def similarity(A, B, metric):
    # rows of A against rows of B (Spearman, Pearson or cosine); NaN where a row is constant
    A, B = np.asarray(A, float), np.asarray(B, float)
    if metric == "spearman":
        A, B = rankdata(A, axis=1), rankdata(B, axis=1)
    if metric in ("spearman", "pearson"):
        A = A - A.mean(axis=1, keepdims=True)
        B = B - B.mean(axis=1, keepdims=True)
    with np.errstate(invalid="ignore", divide="ignore"):
        A = A / np.linalg.norm(A, axis=1, keepdims=True)
        B = B / np.linalg.norm(B, axis=1, keepdims=True)
    return A @ B.T

## Embryo pairs

Cross-species pairs are all mouse × human embryo pairs; a pair is *stage matched* when the stages appear in `STAGE_QUALITY` (good or ambiguous). The baseline pairs are two embryos of the same species and stage. For each pair the genes are selected once (`select_genes` on both embryos) and every shared region is compared with every other, so the matched and mismatched ρ use the same genes.

In [ ]:
KIND_MATCHED, KIND_OTHER, KIND_BASELINE = "cross-species, matched stage", "cross-species, other stage", "same species & stage"
KIND_COLORS = {KIND_MATCHED: C_MATCHED, KIND_OTHER: C_OTHER, KIND_BASELINE: C_BASELINE}


def embryo_pairs():
    pairs = []
    for m in MOUSE:
        for h in HUMAN:
            q = STAGE_QUALITY.get((EMBRYOS[h]["stage"], EMBRYOS[m]["stage"]))
            pairs.append((m, h, KIND_MATCHED if q else KIND_OTHER, q or ""))
    for a, b in combinations(EMBRYOS, 2):
        if EMBRYOS[a]["species"] == EMBRYOS[b]["species"] and EMBRYOS[a]["stage"] == EMBRYOS[b]["stage"]:
            pairs.append((a, b, KIND_BASELINE, ""))
    return pairs


PAIRS = embryo_pairs()
_PAIR_TABLES = {}


def pair_tables(s):
    # per (pair, shared region): rho; per pair: matched vs mismatched rho and the best-match rate
    key = tuple(sorted(s.items()))
    if key in _PAIR_TABLES:
        return _PAIR_TABLES[key]
    rows, summary = [], []
    for a, b, kind, quality in PAIRS:
        keep = select_genes([a, b], s)
        A, B = profiles(a, keep, s), profiles(b, keep, s)
        shared = sorted(set(A.index) & set(B.index))
        if not shared:
            continue
        S = similarity(A.loc[shared], B.loc[shared], s["metric"])
        diag = np.diag(S)
        for r, v in zip(shared, diag):
            rows.append(dict(region=r, a=a, b=b, kind=kind, quality=quality, rho=v))
        off = S[~np.eye(len(shared), dtype=bool)]
        best = (np.nanargmax(S, axis=1) == np.arange(len(shared))) if len(shared) > 1 else np.array([np.nan])
        summary.append(dict(a=a, b=b, kind=kind, quality=quality, genes=int(keep.sum()), regions=len(shared),
                            matched=np.nanmedian(diag), mismatched=np.nanmedian(off) if off.size else np.nan,
                            best_match=np.nanmean(best)))
    out = pd.DataFrame(rows), pd.DataFrame(summary)
    _PAIR_TABLES[key] = out
    return out

## Cross-species gene correlation

Paired observations (stage pair, harmonized region) with one row per observation in a mouse and a human matrix, the gene × gene similarity between them, and the shuffled-pairing null.

In [ ]:
def _cross_settings(s):
    # the cross-species analysis pairs harmonized regions and compares relative profiles
    return dict(s, labels="harmonized", profile="relative")


def cross_candidates(s):
    # genes expressed (>= min CP10k) in some region of *both* species, most variable first (smaller of the two variances)
    P, _ = log_profiles(s["labels"], s["gene_set"])
    thr = np.log1p(s["min_cp10k"])
    expressed, variance = [], []
    for embs in (MOUSE, HUMAN):
        hit = np.zeros(len(GENE_AXIS[s["gene_set"]]), bool)
        for e in embs:
            X = P[e].to_numpy()
            hit |= ((X >= thr) if thr > 0 else (X > 0)).any(axis=0)
        expressed.append(hit)
        variance.append(np.mean([P[e].to_numpy().var(axis=0) for e in embs], axis=0))
    idx = np.flatnonzero(expressed[0] & expressed[1])
    return idx[np.argsort(np.minimum(*variance)[idx])[::-1]]


def stage_groups(pairing):
    # [(name, mouse embryos, human embryos)] whose region means are paired
    if pairing == "all":
        return [("all stages", MOUSE, HUMAN)]
    groups = []
    for h_stage, m_stage in STAGE_QUALITY:
        m = [e for e in MOUSE if EMBRYOS[e]["stage"] == m_stage]
        h = [e for e in HUMAN if EMBRYOS[e]["stage"] == h_stage]
        if m and h:
            groups.append((f"{m_stage}/{h_stage}", m, h))
    if not groups:
        raise ValueError("no stage-matched mouse and human embryos; set Pairing to 'all stages'")
    return groups


def paired_observations(keep, s, pairing):
    # mouse and human matrices (observations x genes) with the same rows: one per (stage pair, shared region)
    M, H, meta = [], [], []
    for b, (name, m, h) in enumerate(stage_groups(pairing)):
        Mg = pd.concat([profiles(e, keep, s) for e in m]).groupby(level=0).mean()
        Hg = pd.concat([profiles(e, keep, s) for e in h]).groupby(level=0).mean()
        shared = sorted(set(Mg.index) & set(Hg.index))
        rows = [f"{name} {r}" for r in shared]
        M.append(Mg.loc[shared].set_axis(rows))
        H.append(Hg.loc[shared].set_axis(rows))
        meta.append(pd.DataFrame({"pair": name, "region": shared, "block": b}, index=rows))
    M, H, meta = pd.concat(M), pd.concat(H), pd.concat(meta)
    if len(M) < 5:
        raise ValueError(f"only {len(M)} paired regions; use more stages or lower 'Min CP10k'")
    return M, H, meta


_CROSS = {}


def cross_matrix(s, n, pairing):
    # top n genes, mouse x human similarity over the paired observations, within-species similarities, null
    s = _cross_settings(s)
    key = (tuple(sorted(s.items())), n, pairing)
    if key not in _CROSS:
        cand = cross_candidates(s)
        if len(cand) < 10:
            raise ValueError(f"only {len(cand)} genes are expressed in both species; lower 'Min CP10k'")
        keep = np.zeros(len(GENE_AXIS[s["gene_set"]]), bool)
        keep[cand[:n]] = True
        M, H, meta = paired_observations(keep, s, pairing)
        Mt, Ht = M.to_numpy().T, H.to_numpy().T
        sim = lambda A, B: np.nan_to_num(similarity(A, B, s["metric"]))
        rng = np.random.default_rng(0)
        blocks = meta.block.to_numpy()
        null_diag, null_all = [], []
        for _ in range(N_PERMUTATIONS):  # shuffle the human regions within each stage pair
            perm = np.arange(len(blocks))
            for b in np.unique(blocks):
                idx = np.flatnonzero(blocks == b)
                perm[idx] = rng.permutation(idx)
            Xp = sim(Mt, Ht[:, perm])
            null_diag.append(np.diag(Xp))
            null_all.append(rng.choice(Xp.ravel(), min(Xp.size, 5000), replace=False))
        _CROSS[key] = dict(s=s, genes=GENE_AXIS[s["gene_set"]][keep], keep=keep, M=M, H=H, meta=meta,
                           X=sim(Mt, Ht), Wm=sim(Mt, Mt), Wh=sim(Ht, Ht),
                           null_diag=np.concatenate(null_diag), null=np.concatenate(null_all))
    return _CROSS[key]


def fmt_p(p):
    return "< 0.001" if p < 0.001 else f"{p:.3f}"


def perm_p(r, null):
    # one-sided empirical p of r against the shuffled pairings
    null = np.sort(null)
    return (1 + len(null) - np.searchsorted(null, np.atleast_1d(r), side="left")) / (1 + len(null))


def gene_modules(C, k):
    # average-linkage clustering on 1 - similarity: leaf order and module per gene (numbered along the order)
    D = np.clip(1 - C, 0, None)
    D = (D + D.T) / 2
    np.fill_diagonal(D, 0)
    Z = linkage(squareform(D, checks=False), "average")
    order = leaves_list(Z)
    raw = fcluster(Z, k, "maxclust")
    number = {}
    for i in order:
        number.setdefault(raw[i], len(number) + 1)
    return order, np.array([number[m] for m in raw])


def _module_blocks(order, modules):
    # (module, first position, last position) along the leaf order
    along = modules[order]
    starts = np.flatnonzero(np.r_[True, along[1:] != along[:-1]])
    ends = np.r_[starts[1:], len(along)] - 1
    return [(int(along[a]), int(a), int(b)) for a, b in zip(starts, ends)]


PAIRINGS = {"stage": "stage matched", "all": "all stages"}


def _cross_line(c, n_obs):
    s = c["s"]
    return (f"{len(c['genes'])} {GENE_SETS[s['gene_set']]} most variable in both species (≥ {s['min_cp10k']:g} CP10k), "
            f"{n_obs} paired regions, {METRIC_NAMES[s['metric']]} of relative profiles")


def cross_figures(s, n, k, pairing):
    c = cross_matrix(s, n, pairing)
    X, M, H, meta = c["X"], c["M"], c["H"], c["meta"]
    order, modules = gene_modules((X + X.T) / 2, k)
    names = np.array(gene_display(c["genes"], s["gene_set"]))
    sim = METRIC_NAMES[s["metric"]]
    q99 = float(np.quantile(c["null"], 0.99))

    heat = go.Figure(go.Heatmap(
        z=X[np.ix_(order, order)], x=names[order], y=names[order], colorscale=DIVERGING, zmid=0, zmin=-1, zmax=1,
        hovertemplate=f"mouse %{{y}}<br>human %{{x}}<br>{sim} = %{{z:.2f}}<extra></extra>",
        colorbar=dict(title=dict(text=sim, side="right"), thickness=12)))
    for m, a, b in _module_blocks(order, modules):
        heat.add_shape(type="rect", x0=a - 0.5, x1=b + 0.5, y0=a - 0.5, y1=b + 0.5, line=dict(color=INK, width=1))
        if b - a >= 4:
            heat.add_annotation(x=(a + b) / 2, y=a - 0.5, text=str(m), showarrow=False, yshift=-8,
                                font=dict(size=10, color=INK), bgcolor="rgba(255,255,255,0.8)")
    heat.update_layout(**{**LAYOUT, "margin": dict(l=10, r=10, t=95, b=10)}, height=840, width=820, plot_bgcolor="white",
                       title=(f"Mouse genes (rows) × human genes (columns), {PAIRINGS[pairing]}, {k} modules"
                              f"<br><sup>{_cross_line(c, len(M))}.<br>Same gene order on both axes; diagonal: the "
                              f"same gene in both species; 99% of shuffled pairings are below {q99:.2f}</sup>"),
                       xaxis_title="gene in human", yaxis_title="gene in mouse")
    heat.update_xaxes(showticklabels=False, showgrid=False)
    heat.update_yaxes(showticklabels=False, showgrid=False, autorange="reversed", scaleanchor="x")

    bins = np.linspace(-1, 1, 41)
    centers = (bins[:-1] + bins[1:]) / 2
    hist = go.Figure()
    diag = np.diag(X)
    for label, v, color in (("same gene (diagonal)", diag, C_MATCHED),
                            ("different genes (off-diagonal)", X[~np.eye(len(X), dtype=bool)], C_OTHER),
                            ("shuffled pairings", c["null"], C_BASELINE)):
        dens, _ = np.histogram(v, bins=bins, density=True)
        hist.add_trace(go.Scatter(x=centers, y=dens, mode="lines", line=dict(shape="hvh", width=2, color=color),
                                  name=f"{label}, median {np.median(v):.2f}",
                                  hovertemplate=f"{label}<br>{sim} %{{x:.2f}}<br>density %{{y:.2f}}<extra></extra>"))
    hist.add_vline(x=q99, line=dict(color=MUTED, width=1, dash="dash"),
                   annotation_text="99% of shuffled", annotation_position="top")
    hist.update_layout(**LAYOUT, height=380, width=820, xaxis_title=sim, yaxis_title="density",
                       title="How strong are the cross-species correlations?",
                       legend=dict(orientation="h", y=-0.25))

    regions_m = M.groupby(meta.region).mean()
    regions_h = H.groupby(meta.region).mean()
    rows = []
    for m in range(1, modules.max() + 1):
        g = modules == m
        sub = X[np.ix_(g, g)]
        hub = (sub.mean(axis=0) + sub.mean(axis=1)) / 2
        rows.append({"module": m, "genes": int(g.sum()), "cross r": sub.mean(), "same gene r": np.diag(sub).mean(),
                     "peak region mouse": regions_m.loc[:, g].mean(axis=1).idxmax(),
                     "peak region human": regions_h.loc[:, g].mean(axis=1).idxmax(),
                     "hub genes": ", ".join(names[g][np.argsort(hub)[::-1][:6]])})
    modules_table = pd.DataFrame(rows).set_index("module").round(3)

    same = pd.DataFrame({"gene": names, "r": diag.round(3), "p (shuffled)": [fmt_p(v) for v in perm_p(diag, c["null_diag"])]})
    same = same.sort_values("r", ascending=False)
    off = X.copy()
    np.fill_diagonal(off, -np.inf)
    flat = np.argsort(off.ravel())[::-1][:N_PAIRS]
    i, j = np.unravel_index(flat, X.shape)
    pairs = pd.DataFrame({"mouse gene": names[i], "human gene": names[j], "cross r": X[i, j],
                          "within mouse": c["Wm"][i, j], "within human": c["Wh"][i, j],
                          "same gene r (mouse gene)": diag[i], "same gene r (human gene)": diag[j]})
    return heat, hist, modules_table, same, pairs.round(3), q99

## Plot helpers

In [ ]:
METRIC_NAMES = {"spearman": "Spearman ρ", "pearson": "Pearson r", "cosine": "cosine similarity"}
LAYOUT = dict(template="plotly_white", font=dict(color=INK), margin=dict(l=10, r=10, t=70, b=10))


def _metric_label(s):
    return f"{METRIC_NAMES[s['metric']]}, {s['profile']} profiles"


def _settings_line(s, n_genes=None):
    genes = f"{n_genes:,} genes, " if n_genes is not None else ""
    top = f"top {s['top_variable']:,} variable" if s["top_variable"] else "all variable"
    return (f"{genes}{GENE_SETS[s['gene_set']]} ({top}, ≥ {s['min_cp10k']:g} CP10k), "
            f"{s['labels']} labels, {_metric_label(s)}")


def _color_axis(S, s):
    # diverging around 0 for relative profiles, sequential for absolute ones
    if s["profile"] == "relative":
        lim = float(np.nanmax(np.abs(S))) if np.isfinite(S).any() else 1.0
        return dict(colorscale=DIVERGING, zmid=0, zmin=-lim, zmax=lim)
    return dict(colorscale=SEQUENTIAL, zmin=float(np.nanmin(S)), zmax=float(np.nanmax(S)))


def species_label(e):
    return f"{EMBRYOS[e]['species']} {e}"


def matched_regions_plot(s):
    rows, _ = pair_tables(s)
    if rows.empty:
        raise ValueError("no region is shared by any embryo pair with these settings")
    cross = rows[rows.kind == KIND_MATCHED]
    if cross.empty:
        cross = rows[rows.kind != KIND_BASELINE]
    order = cross.groupby("region").rho.median().sort_values(ascending=False).index.tolist()
    order += sorted(set(rows.region) - set(order))
    fig = go.Figure()
    for kind in (KIND_MATCHED, KIND_OTHER, KIND_BASELINE):
        d = rows[rows.kind == kind]
        if d.empty:
            continue
        text = [f"{species_label(a)} vs {species_label(b)}" + (f" (stages: {q})" if q else "")
                for a, b, q in zip(d.a, d.b, d.quality)]
        color = KIND_COLORS[kind]
        fig.add_trace(go.Box(
            x=d.region, y=d.rho, name=f"{kind} ({d[['a', 'b']].drop_duplicates().shape[0]} pairs)",
            boxpoints="all", jitter=0.5, pointpos=0, text=text, hoveron="points",
            hovertemplate="%{x}<br>%{text}<br>ρ = %{y:.3f}<extra></extra>",
            marker=dict(size=8, color=color, line=dict(width=1, color="white")),
            line=dict(width=1, color=color), fillcolor="rgba(0,0,0,0)",
        ))
    fig.update_layout(**LAYOUT, boxmode="group", height=560,
                      title=f"Same region, different embryos<br><sup>{_settings_line(s)}</sup>",
                      yaxis_title=_metric_label(s), legend=dict(orientation="h", y=-0.18))
    fig.update_xaxes(categoryorder="array", categoryarray=order)
    fig.add_hline(y=0, line=dict(color="#c3c2b7", width=1))
    return fig


def matched_regions_table(s):
    rows, _ = pair_tables(s)
    t = rows.pivot_table(index="region", columns="kind", values="rho", aggfunc="median")
    t = t.reindex(columns=[k for k in (KIND_MATCHED, KIND_OTHER, KIND_BASELINE) if k in t.columns])
    if KIND_MATCHED in t and KIND_BASELINE in t:
        t["matched − baseline"] = t[KIND_MATCHED] - t[KIND_BASELINE]
    sort = KIND_MATCHED if KIND_MATCHED in t else t.columns[0]
    return t.sort_values(sort, ascending=False).round(3)


def one_pair(mouse, human):
    if ALL_EMBRYOS in (mouse, human):
        raise ValueError("pick one mouse and one human embryo for this view (all embryos: gene scatter with All regions)")


def region_heatmap(mouse, human, s):
    one_pair(mouse, human)
    keep = select_genes([mouse, human], s)
    A, B = profiles(mouse, keep, s), profiles(human, keep, s)
    _, bins = log_profiles(s["labels"], s["gene_set"])
    shared = sorted(set(A.index) & set(B.index))
    rows = shared + sorted(set(A.index) - set(shared))
    cols = shared + sorted(set(B.index) - set(shared))
    S = similarity(A.loc[rows], B.loc[cols], s["metric"])
    text = np.full(S.shape, "", dtype=object)  # label only each row's best match
    for i in range(len(rows)):
        if np.isfinite(S[i]).any():
            j = int(np.nanargmax(S[i]))
            text[i, j] = f"{S[i, j]:.2f}"
    custom = np.dstack([np.repeat(bins[mouse].loc[rows].to_numpy()[:, None], len(cols), 1),
                        np.repeat(bins[human].loc[cols].to_numpy()[None, :], len(rows), 0)])
    fig = go.Figure(go.Heatmap(
        z=S, x=cols, y=rows, text=text, texttemplate="%{text}", textfont=dict(size=10), customdata=custom,
        hovertemplate=(f"mouse %{{y}} (%{{customdata[0]:,}} bins)<br>human %{{x}} (%{{customdata[1]:,}} bins)"
                       f"<br>{METRIC_NAMES[s['metric']]} = %{{z:.3f}}<extra></extra>"),
        colorbar=dict(title=dict(text=METRIC_NAMES[s["metric"]], side="right"), thickness=12),
        xgap=2, ygap=2, **_color_axis(S, s),
    ))
    for i in range(len(shared)):  # outline the same-name cells
        fig.add_shape(type="rect", x0=i - 0.5, x1=i + 0.5, y0=i - 0.5, y1=i + 0.5,
                      line=dict(color=INK, width=1.5))
    size = 34
    fig.update_layout(**LAYOUT, height=180 + size * len(rows), width=max(820, 260 + size * len(cols)),
                      title=(f"Mouse {mouse} (rows) vs human {human} (columns)"
                             f"<br><sup>{_settings_line(s, int(keep.sum()))}. Outlined: same region; "
                             f"number: best match of the row</sup>"),
                      plot_bgcolor="white")
    fig.update_xaxes(side="bottom", tickangle=-45, showgrid=False)
    fig.update_yaxes(autorange="reversed", showgrid=False)
    return fig


def embryo_heatmap(region, s):
    if region == ALL_REGIONS:
        raise ValueError("pick a single region for this tab")
    _, bins = log_profiles(s["labels"], s["gene_set"])
    embs = [e for e in MOUSE + HUMAN if region in bins[e].index]
    if len(embs) < 2:
        raise ValueError(f"'{region}' has a profile in fewer than two embryos")
    keep = select_genes(embs, s)
    X = np.vstack([profiles(e, keep, s).loc[region].to_numpy() for e in embs])
    S = similarity(X, X, s["metric"])
    order = np.arange(len(embs))
    if len(embs) > 2 and np.isfinite(S).all():
        D = np.clip(1 - S, 0, None)
        np.fill_diagonal(D, 0)
        order = leaves_list(linkage(squareform((D + D.T) / 2, checks=False), "average"))
    names = [f"{species_label(embs[i])} ({bins[embs[i]][region]:,} bins)" for i in order]
    S = S[np.ix_(order, order)]
    fig = go.Figure(go.Heatmap(
        z=S, x=names, y=names, texttemplate="%{z:.2f}", textfont=dict(size=10),
        hovertemplate=f"%{{y}}<br>%{{x}}<br>{METRIC_NAMES[s['metric']]} = %{{z:.3f}}<extra></extra>",
        colorbar=dict(title=dict(text=METRIC_NAMES[s["metric"]], side="right"), thickness=12),
        xgap=2, ygap=2, **_color_axis(S[~np.eye(len(S), dtype=bool)], s),
    ))
    size = 52
    fig.update_layout(**LAYOUT, height=260 + size * len(embs), width=420 + size * len(embs),
                      title=f"{region} in every embryo (clustered)<br><sup>{_settings_line(s, int(keep.sum()))}</sup>",
                      plot_bgcolor="white")
    fig.update_xaxes(tickangle=-45, showgrid=False)
    fig.update_yaxes(autorange="reversed", showgrid=False)
    return fig


def gene_display(keys, gene_set):
    mouse = MOUSE_SYMBOL[gene_set]
    return [k if mouse.get(k, k).upper() == k else f"{k} / {mouse[k]}" for k in keys]


def gene_scatter(mouse, human, region, s):
    one_pair(mouse, human)
    _, bins = log_profiles(s["labels"], s["gene_set"])
    missing = [e for e in (mouse, human) if region not in bins[e].index]
    if missing:
        raise ValueError(f"'{region}' has no profile in {', '.join(missing)} (absent or < {MIN_BINS} bins)")
    keep = select_genes([mouse, human], s)
    x = profiles(mouse, keep, s).loc[region]
    y = profiles(human, keep, s).loc[region]
    rho = similarity(x.to_numpy()[None], y.to_numpy()[None], s["metric"])[0, 0]
    names = np.array(gene_display(x.index, s["gene_set"]))
    df = pd.DataFrame({"gene": names, "mouse": x.to_numpy(), "human": y.to_numpy()})
    df["human − mouse"] = df.human - df.mouse
    df["both"] = np.minimum(df.mouse, df.human)
    groups = {
        "high in both": (df.nlargest(N_TABLE, "both"), C_MATCHED),
        "higher in mouse": (df.nsmallest(N_TABLE, "human − mouse"), C_OTHER),
        "higher in human": (df.nlargest(N_TABLE, "human − mouse"), C_HUMAN_UP),
    }
    unit = "log1p CP10k − embryo mean" if s["profile"] == "relative" else "log1p CP10k"
    fig = go.Figure(go.Scattergl(
        x=df.mouse, y=df.human, mode="markers", name=f"genes ({len(df):,})", text=df.gene,
        marker=dict(size=5, color=C_GENE, opacity=0.6),
        hovertemplate="%{text}<br>mouse %{x:.2f}<br>human %{y:.2f}<extra></extra>",
    ))
    for label, (d, color) in groups.items():
        top = d.head(N_LABELLED)
        fig.add_trace(go.Scatter(
            x=d.mouse, y=d.human, mode="markers", name=f"{label} (top {len(d)})", text=d.gene,
            marker=dict(size=8, color=color, line=dict(width=1, color="white")),
            hovertemplate="%{text}<br>mouse %{x:.2f}<br>human %{y:.2f}<extra></extra>",
        ))
        for k, (g, gx, gy) in enumerate(zip(top.gene, top.mouse, top.human)):  # alternate sides so neighbours don't overlap
            fig.add_annotation(x=gx, y=gy, text=g, showarrow=True, arrowwidth=1, arrowcolor=MUTED, arrowhead=0,
                               ax=30 if k % 2 else -30, ay=-14 if k % 4 < 2 else 14, font=dict(size=10, color=INK))
    lo = float(min(df.mouse.min(), df.human.min()))
    hi = float(max(df.mouse.max(), df.human.max()))
    fig.add_shape(type="line", x0=lo, y0=lo, x1=hi, y1=hi, line=dict(color=MUTED, width=1, dash="dash"), layer="below")
    fig.update_layout(**LAYOUT, height=640, width=760,
                      title=(f"{region}: mouse {mouse} vs human {human}, {METRIC_NAMES[s['metric']]} = {rho:.3f}"
                             f"<br><sup>{_settings_line(s, len(df))}</sup>"),
                      xaxis_title=f"mouse ({unit})", yaxis_title=f"human ({unit})",
                      legend=dict(orientation="h", y=-0.15))
    fig.update_yaxes(scaleanchor="x", scaleratio=1)
    tables = {k: d.drop(columns="both").set_index("gene").round(3) for k, (d, _) in groups.items()}
    return fig, tables


def rowwise_similarity(A, B, metric):
    # row i of A against row i of B (Spearman, Pearson or cosine); NaN where a row is constant
    A, B = np.asarray(A, float), np.asarray(B, float)
    if metric == "spearman":
        A, B = rankdata(A, axis=1), rankdata(B, axis=1)
    if metric in ("spearman", "pearson"):
        A = A - A.mean(axis=1, keepdims=True)
        B = B - B.mean(axis=1, keepdims=True)
    with np.errstate(invalid="ignore", divide="ignore"):
        return (A * B).sum(axis=1) / (np.linalg.norm(A, axis=1) * np.linalg.norm(B, axis=1))


def block_permutation(blocks, rng):
    # a permutation that only moves elements within their block
    perm = np.arange(len(blocks))
    for b in np.unique(blocks):
        idx = np.flatnonzero(blocks == b)
        perm[idx] = rng.permutation(idx)
    return perm


def _observations(mouse, human, s, component):
    # genes x observations for mouse and human, block labels for the shuffles, peak labels and a description
    if mouse != ALL_EMBRYOS:
        one_pair(mouse, human)
        keep = select_genes([mouse, human], s)
        A, B = profiles(mouse, keep, s), profiles(human, keep, s)
        shared = sorted(set(A.index) & set(B.index))
        if len(shared) < 4:
            raise ValueError(f"{mouse} and {human} share only {len(shared)} regions; at least 4 are needed")
        info = dict(title=f"mouse {mouse} vs human {human}, {len(shared)} shared regions", source="",
                    what=f"the {len(shared)} regions both embryos share ({', '.join(shared)})", shuffled="region orders")
        return (A.loc[shared].T, B.loc[shared].T, np.zeros(len(shared), int),
                np.array(shared), np.array(shared), info)
    M, H, meta = paired_observations(select_genes(MOUSE + HUMAN, s), s, "stage")
    m_stage = meta.pair.str.split("/").str[0]
    h_stage = meta.pair.str.split("/").str[1]
    pairs = list(dict.fromkeys(meta.pair))
    if component == "regions":  # same regional pattern within each stage: remove each stage's mean
        M, H = M - M.groupby(meta.pair).transform("mean"), H - H.groupby(meta.pair).transform("mean")
        blocks, shuffled = meta.pair.to_numpy(), "regions within each stage"
    elif component == "time":  # same change over stages within each region: remove each region's mean
        multi = meta.region.map(meta.region.value_counts()).to_numpy() >= 2
        M, H, meta, m_stage, h_stage = M[multi], H[multi], meta[multi], m_stage[multi], h_stage[multi]
        if len(meta) < 4:
            raise ValueError("too few regions are present at two or more stage pairs")
        M, H = M - M.groupby(meta.region).transform("mean"), H - H.groupby(meta.region).transform("mean")
        blocks, shuffled = meta.region.to_numpy(), "stages within each region"
    else:
        blocks, shuffled = np.zeros(len(meta), int), "observations"
    info = dict(title=f"all embryos, {COMPONENTS[component]}, {len(meta)} paired observations",
                source=f"stage pairs {', '.join(pairs)}; ",
                what=(f"{len(meta)} (stage pair, region) observations from {', '.join(pairs)}; each is the mean of "
                      f"that stage's embryos" + {"all": "", "regions": ", centred within each stage (time removed)",
                                                  "time": ", centred within each region (regions removed); only regions "
                                                          "present at two or more stage pairs"}[component]),
                shuffled=shuffled)
    return (M.T, H.T, blocks, (m_stage + " " + meta.region).to_numpy(), (h_stage + " " + meta.region).to_numpy(), info)


def gene_region_correlation(mouse, human, s, component="all"):
    # per gene: similarity of its profile over the paired observations, mouse vs human, with a shuffled null
    A, B, blocks, peak_m, peak_h, info = _observations(mouse, human, s, component)
    Am, Bh = A.to_numpy(), B.to_numpy()  # genes x observations
    spread_m, spread_h = Am.std(axis=1), Bh.std(axis=1)
    ok = (spread_m > 1e-12) & (spread_h > 1e-12)
    Am, Bh = Am[ok], Bh[ok]
    r = rowwise_similarity(Am, Bh, s["metric"])
    rng = np.random.default_rng(0)
    null = np.concatenate([rowwise_similarity(Am, Bh[:, block_permutation(blocks, rng)], s["metric"])
                           for _ in range(N_REGION_SHUFFLES)])
    null = np.sort(null[np.isfinite(null)])
    lo, hi = np.quantile(null, [0.025, 0.975])
    upper = 1 - np.searchsorted(null, r, side="left") / len(null)   # P(null >= r)
    lower = np.searchsorted(null, r, side="right") / len(null)      # P(null <= r)
    p = np.clip(2 * np.minimum(upper, lower), 1 / len(null), 1)
    df = pd.DataFrame({
        "gene": gene_display(A.index[ok], s["gene_set"]), "r": r, "p": p,
        "spread": np.minimum(spread_m[ok], spread_h[ok]), "spread mouse": spread_m[ok], "spread human": spread_h[ok],
        "peak mouse": peak_m[Am.argmax(axis=1)], "peak human": peak_h[Bh.argmax(axis=1)],
    }).dropna(subset=["r"])
    df["kind"] = np.where(df.r > hi, "positive", np.where(df.r < lo, "negative", "neither"))
    return df, (lo, hi), info


def gene_correlation_plot(mouse, human, s, component="all"):
    df, (lo, hi), info = gene_region_correlation(mouse, human, s, component)
    sim = METRIC_NAMES[s["metric"]]
    counts = df.kind.value_counts()
    fig = go.Figure()
    for kind, color in (("neither", C_GENE), ("positive", C_MATCHED), ("negative", C_OTHER)):
        d = df[df.kind == kind]
        big = kind != "neither"
        fig.add_trace((go.Scatter if big else go.Scattergl)(
            x=d.r, y=d.spread, mode="markers", name=f"{kind} ({len(d):,})", text=d.gene,
            customdata=np.c_[[fmt_p(v) for v in d.p], d["peak mouse"], d["peak human"]],
            marker=dict(size=7 if big else 5, color=color, opacity=0.85 if big else 0.6,
                        line=dict(width=1 if big else 0, color="white")),
            hovertemplate=(f"%{{text}}<br>{sim} = %{{x:.2f}} (p %{{customdata[0]}})<br>spread %{{y:.2f}}"
                           "<br>peak: mouse %{customdata[1]}, human %{customdata[2]}<extra></extra>")))
        if big:  # label the strongest patterns: large |r| on genes that vary a lot
            top = d.assign(score=d.spread * d.r.abs()).nlargest(N_LABELLED, "score")
            for k, (g, gx, gy) in enumerate(zip(top.gene, top.r, top.spread)):
                fig.add_annotation(x=gx, y=gy, text=g, showarrow=True, arrowwidth=1, arrowcolor=MUTED, arrowhead=0,
                                   ax=30 if k % 2 else -30, ay=-14 if k % 4 < 2 else 14, font=dict(size=10, color=INK))
    for x in (lo, hi):
        fig.add_vline(x=x, line=dict(color=MUTED, width=1, dash="dash"))
    unit = "log1p CP10k" + (" − embryo mean" if s["profile"] == "relative" else "")
    pattern = {"all": "pattern", "regions": "regional pattern", "time": "change over time"}[component]
    fig.update_layout(**{**LAYOUT, "margin": dict(l=10, r=10, t=120, b=10)}, height=660, width=820,
                      title=(f"All regions: {info['title']}"
                             f"<br><sup>{counts.get('positive', 0):,} genes follow the same {pattern}, "
                             f"{counts.get('negative', 0):,} the opposite<br>{info['source']}dashed: 95% of "
                             f"{N_REGION_SHUFFLES} shuffles ({info['shuffled']})<br>{_settings_line(s, len(df))}</sup>"),
                      xaxis_title=f"{sim} of the gene, mouse vs human",
                      yaxis_title=f"spread (smaller SD of the two species, {unit})",
                      legend=dict(orientation="h", y=-0.15))
    fig.update_xaxes(range=[-1.05, 1.05])
    cols = ["gene", "r", "p", "spread mouse", "spread human", "peak mouse", "peak human"]
    table = lambda d: d[cols].assign(p=[fmt_p(v) for v in d.p]).set_index("gene").round(3)
    tables = {f"positive: same {pattern} in both species": table(df[df.kind == "positive"].nlargest(N_TABLE, "r")),
              f"negative: opposite {pattern}": table(df[df.kind == "negative"].nsmallest(N_TABLE, "r"))}
    return fig, tables, info


def pair_summary(s):
    _, summary = pair_tables(s)
    t = summary.copy()
    t.insert(0, "pair", [f"{species_label(a)} vs {species_label(b)}" for a, b in zip(t.a, t.b)])
    t["matched − mismatched"] = t.matched - t.mismatched
    t = t.drop(columns=["a", "b"]).sort_values(["kind", "matched"], ascending=[True, False])
    return t.set_index("pair").round(3)

## Viewer

The settings in the top rows apply to every tab. *Mouse* and *Human* pick the embryo pair for the region × region and gene-scatter tabs, *Region* the region for the embryo × embryo and gene-scatter tabs (*All regions*: gene scatter only). *All embryos* (in Mouse or Human, which sets both) works with gene scatter + *All regions* and enables *Component*. *Cross genes*, *Modules* and *Pairing* apply to the cross-species tab. Only the visible tab is computed; results are cached, so switching back is instant.

In [ ]:
STATE = {"busy": False}  # `busy` pauses redraws while widgets are set


class _pause:
    def __enter__(self):
        STATE["busy"] = True

    def __exit__(self, *exc):
        STATE["busy"] = False


gene_set_dd = w.Dropdown(options=[(v, k) for k, v in GENE_SETS.items()], value=DEFAULT_GENE_SET,
                         description="Gene set", layout=w.Layout(width="260px"))
profile_tg = w.ToggleButtons(options=[("relative", "relative"), ("absolute", "absolute")], description="Profile")
metric_dd = w.Dropdown(options=[(v, k) for k, v in METRIC_NAMES.items()], value="spearman",
                       description="Similarity", layout=w.Layout(width="260px"))
labels_tg = w.ToggleButtons(options=[("harmonized", "harmonized"), ("original", "original")], description="Labels")
top_dd = w.Dropdown(options=[("all", 0), ("500", 500), ("1,000", 1000), ("2,000", 2000), ("5,000", 5000)],
                    value=DEFAULT_TOP_VARIABLE, description="Top variable", layout=w.Layout(width="200px"))
min_dd = w.Dropdown(options=[("0 (> 0)", 0.0), ("0.1", 0.1), ("0.5", 0.5), ("1", 1.0), ("5", 5.0)],
                    value=DEFAULT_MIN_CP10K, description="Min CP10k", layout=w.Layout(width="200px"))
mouse_dd = w.Dropdown(options=[(ALL_EMBRYOS, ALL_EMBRYOS)] + [(species_label(e), e) for e in MOUSE], value=MOUSE[0],
                      description="Mouse", layout=w.Layout(width="260px"))
human_dd = w.Dropdown(options=[(ALL_EMBRYOS, ALL_EMBRYOS)] + [(species_label(e), e) for e in HUMAN], value=HUMAN[0],
                      description="Human", layout=w.Layout(width="260px"))
component_tg = w.ToggleButtons(options=[(v, k) for k, v in COMPONENTS.items()], description="Component",
                               layout=w.Layout(display="none"))
region_dd = w.Dropdown(description="Region", layout=w.Layout(width="260px"))
cross_dd = w.Dropdown(options=[(f"{n:,}", n) for n in CROSS_GENES], value=DEFAULT_CROSS_GENES,
                      description="Cross genes", layout=w.Layout(width="200px"))
modules_sl = w.IntSlider(value=DEFAULT_MODULES, min=2, max=20, description="Modules", layout=w.Layout(width="260px"))
pairing_tg = w.ToggleButtons(options=[(v, k) for k, v in PAIRINGS.items()], description="Pairing")
status = w.HTML()

TAB_TITLES = ["Matched regions", "Region × region", "Embryo × embryo", "Gene scatter", "Pair summary",
              "Cross-species genes"]
outputs = [w.Output() for _ in TAB_TITLES]
tabs = w.Tab(children=outputs, titles=TAB_TITLES)


def settings():
    return dict(gene_set=gene_set_dd.value, profile=profile_tg.value, metric=metric_dd.value,
                labels=labels_tg.value, top_variable=top_dd.value, min_cp10k=min_dd.value)


def update_regions():
    # regions with a profile in at least one embryo under the current labels, most common first
    _, bins = log_profiles(labels_tg.value, gene_set_dd.value)
    counts = pd.Series([r for e in EMBRYOS for r in bins[e].index]).value_counts()
    regions = sorted(counts.index, key=lambda r: (-counts[r], r))
    regions = [ALL_REGIONS] + regions
    keep = region_dd.value if region_dd.value in regions else ("Heart" if "Heart" in regions else regions[1])
    with _pause():
        region_dd.options = regions
        region_dd.value = keep


def draw_matched(s):
    display(matched_regions_plot(s))
    display(HTML("<b>Median per region and kind of pair.</b> <code>matched − baseline</code> &lt; 0 means mouse and "
                 "human agree less than two embryos of one species do."))
    display(matched_regions_table(s))


def draw_region_heatmap(s):
    display(region_heatmap(mouse_dd.value, human_dd.value, s))


def draw_embryo_heatmap(s):
    display(embryo_heatmap(region_dd.value, s))


def draw_gene_scatter(s):
    if region_dd.value == ALL_REGIONS:
        fig, tables, info = gene_correlation_plot(mouse_dd.value, human_dd.value, s, component_tg.value)
        display(fig)
        display(HTML(
            f"<div style='max-width:820px;line-height:1.45'>Each gene's profile over {info['what']} is compared between "
            "mouse and human. Positive: high and low in the same places in both species; negative: the opposite. "
            "<code>spread</code>: standard deviation over those observations in each species; <code>peak</code>: where "
            f"the gene is highest (for <i>regions</i> and <i>time</i>: relative to its stage or region mean). A gene counts as positive or negative beyond the {N_REGION_SHUFFLES} shuffles of "
            f"{info['shuffled']}. With <i>all embryos</i>, <code>Profile = relative</code> removes each embryo's overall "
            "level, so <i>time</i> only shows region-specific changes; <code>absolute</code> keeps whole-embryo changes."
            "</div>"))
        _side_by_side(tables)
        return
    fig, tables = gene_scatter(mouse_dd.value, human_dd.value, region_dd.value, s)
    display(fig)
    parts = [f"<div style='margin-right:24px'><b>{k}</b>{t.to_html(classes='gene-table')}</div>" for k, t in tables.items()]
    display(HTML("<style>.gene-table td, .gene-table th {padding: 1px 8px; text-align: right}</style>"
                 "<div style='display:flex;flex-wrap:wrap'>" + "".join(parts) + "</div>"))


def draw_pair_summary(s):
    display(HTML(
        "<div style='max-width:760px;line-height:1.45'><b>Columns.</b> <code>genes</code>: genes compared; "
        "<code>regions</code>: regions present in both embryos; <code>matched</code>: median similarity of a region "
        "with the same region; <code>mismatched</code>: median over all pairs of different regions; "
        "<code>best_match</code>: fraction of regions whose most similar region in the other embryo is the one of the "
        "same name (1.0 = every region finds its partner).</div>"))
    display(pair_summary(s))


def _side_by_side(tables):
    parts = [f"<div style='margin-right:24px'><b>{k}</b>{t.to_html(classes='gene-table')}</div>" for k, t in tables.items()]
    display(HTML("<style>.gene-table td, .gene-table th {padding: 1px 8px; text-align: right}</style>"
                 "<div style='display:flex;flex-wrap:wrap;align-items:flex-start'>" + "".join(parts) + "</div>"))


def draw_cross(s):
    heat, hist, modules_table, same, pairs, q99 = cross_figures(s, cross_dd.value, modules_sl.value, pairing_tg.value)
    display(heat)
    display(hist)
    display(HTML(
        "<div style='max-width:900px;line-height:1.45'><b>Modules</b> (outlined in the matrix). "
        "<code>cross r</code>: mean similarity between the module's genes in mouse and in human; "
        "<code>same gene r</code>: mean of its diagonal; <code>peak region</code>: where the module's genes are "
        "highest in each species; <code>hub genes</code>: the genes most similar to the rest of the module across "
        "species.</div>"))
    display(modules_table)
    _side_by_side({"same gene, most alike": same.head(N_TABLE).set_index("gene").round(3),
                   "same gene, least alike": same.tail(N_TABLE)[::-1].set_index("gene").round(3)})
    display(HTML(
        f"<div style='max-width:900px;line-height:1.45'><b>Top cross pairs:</b> different genes that behave alike "
        f"across species (99% of shuffled pairings are below {q99:.2f}). <code>within mouse</code> / "
        "<code>within human</code>: the same two genes within one species. High in both means the pair is co-expressed "
        "in either species anyway; a high <code>cross r</code> with low within-species values is a cross-species "
        "match only. <code>same gene r</code>: the diagonal value of each gene.</div>"))
    display(pairs)


DRAWERS = [draw_matched, draw_region_heatmap, draw_embryo_heatmap, draw_gene_scatter, draw_pair_summary,
           draw_cross]


def refresh(*_):
    if STATE["busy"]:
        return
    s = settings()
    i = tabs.selected_index
    gene_all = TAB_TITLES[i] == "Gene scatter" and region_dd.value == ALL_REGIONS and mouse_dd.value == ALL_EMBRYOS
    component_tg.layout.display = "" if gene_all else "none"
    with outputs[i]:
        clear_output(wait=True)
        status.value = "Computing …"
        t0 = time.time()
        try:
            DRAWERS[i](s)
        except ValueError as e:
            print(e)
        status.value = f"{TAB_TITLES[i]}: {time.time() - t0:.1f}s"


def on_labels(*_):
    update_regions()
    refresh()


def on_embryo(change):
    # all embryos is a setting for both species: choosing it in one dropdown sets the other, leaving it resets the other
    other = human_dd if change["owner"] is mouse_dd else mouse_dd
    with _pause():
        if change["new"] == ALL_EMBRYOS:
            other.value = ALL_EMBRYOS
        elif other.value == ALL_EMBRYOS:
            other.value = other.options[1][1]
    refresh()


for widget in (gene_set_dd, profile_tg, metric_dd, top_dd, min_dd, region_dd, component_tg,
               cross_dd, modules_sl, pairing_tg):
    widget.observe(refresh, names="value")
for widget in (mouse_dd, human_dd):
    widget.observe(on_embryo, names="value")
labels_tg.observe(on_labels, names="value")
tabs.observe(refresh, names="selected_index")

ui = w.VBox([
    w.HBox([gene_set_dd, top_dd, min_dd]),
    w.HBox([profile_tg]),
    w.HBox([labels_tg]),
    w.HBox([metric_dd]),
    w.HBox([mouse_dd, human_dd, region_dd]),
    w.HBox([component_tg]),
    w.HBox([cross_dd, modules_sl, pairing_tg]),
    status,
    tabs,
])
update_regions()
display(ui)
refresh()